# Notebook 09 — Generazione del modello bayesiano source (digits, SVHN)

Stessa filosofia del Notebook 03 (`f = h ∘ g`, solo `h` riceve il trattamento
bayesiano), applicata qui a SVHN come source invece di MNIST: SVHN (foto reali
di numeri civici) è scelto per lo shift più marcato verso MNIST/USPS (scansioni
pulite) -- uno scenario di adattamento più informativo di MNIST↔USPS, il cui
source-only è già troppo alto per lasciare margine a un adattamento di mostrare
un effetto.

Il training vero e proprio (MAP + early stopping, necessario per SVHN --
diversamente da MNIST nel Notebook 03, che converge in 3 epoche fisse con
`train_map`, SVHN overfitta se non fermato su una validazione) vive in
`code_v2/src/digits_train.py`, non qui. La cella di setup sotto lo lancia
automaticamente **solo se manca ancora un checkpoint**; se il checkpoint esiste
già, lo salta e stampa un avviso, senza riaddestrare.

**Costo del training e determinismo.** Su questa macchina (GTX 1650, CUDA,
`torch 2.5.1+cu121`) il training completo è di ~50 secondi, non i ~15-25 minuti
indicati da una versione precedente di questa cella, scritta quando il device di
riferimento era la MPS di Apple: `resolve_device` sceglie CUDA per prima, e il
divario fra i due backend su questa architettura è di due ordini di grandezza.
`train_source_model` fissa anche `cudnn.deterministic=True` /
`cudnn.benchmark=False`: senza, i soli seed non bastano su CUDA e due run con lo
stesso `seed=2019` divergono di ~3pp sull'accuracy target (misurato: mnist
60.55% vs 57.13%) -- una deriva dello stesso ordine della varianza fra seed che
il Notebook 15 (`15_digits_multiseed.ipynb`) vuole misurare.

Per verificare esplicitamente che il checkpoint ricaricato riproduca l'accuracy
riportata a fine training:

```bash
python code_v2/src/digits_verify.py
```

Questo notebook carica il checkpoint e fa il fit di Laplace sull'ultimo layer --
stesso schema di come i Notebook 04 e 5 caricano gli artefatti salvati dal
Notebook 03.

## Setup

In [1]:
import sys
from pathlib import Path

here = Path().resolve()
for base in [here, *here.parents]:
    if (base / "code_v2" / "src" / "laplace_core.py").is_file():
        sys.path.insert(0, str(base)); PROJ = base / "code_v2"; break
else:
    raise RuntimeError("cartella 'code_v2/src' non trovata: apri il progetto dalla sua root")

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

from code_v2.src.digits_data import load_domain
from code_v2.src.digits_model import SmallCNN32
from code_v2.src.bayesian_model import extract, head_weights, augment, LastLayerLaplace

CHECKPOINT_PATH = PROJ / "models" / "source_svhn" / "model.pt"
SOURCE_DOMAIN = "svhn"
TARGET_DOMAINS = ["mnist", "usps"]
BATCH_SIZE = 128

if not CHECKPOINT_PATH.exists():
    print(f"{CHECKPOINT_PATH} non trovato -- lancio il training (code_v2/src/digits_train.py)...")
    from code_v2.src.digits_train import main as train_digits_main
    train_digits_main()
else:
    print(f"{CHECKPOINT_PATH} trovato -- training saltato")

ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
mean, std = ckpt["source_mean"], ckpt["source_std"]
WEIGHT_DECAY = ckpt["weight_decay"]
N_SOURCE_TRAIN = ckpt["n_source_train"]
print(f"checkpoint: {CHECKPOINT_PATH}")
print(f"source_mean={mean:.4f} source_std={std:.4f} weight_decay={WEIGHT_DECAY} "
      f"n_source_train={N_SOURCE_TRAIN}")

C:\Users\aless\PycharmProjects\BayesianExoAdaptation\code_v2\models\source_svhn\model.pt trovato -- training saltato
checkpoint: C:\Users\aless\PycharmProjects\BayesianExoAdaptation\code_v2\models\source_svhn\model.pt
source_mean=0.4453 source_std=0.1970 weight_decay=0.001 n_source_train=65932


## 1. Caricamento del source model verificato + domini target

`code_v2/src/digits_verify.py` ha già confermato che questo checkpoint, ricaricato
da zero, riproduce esattamente l'accuracy riportata a fine training (PASS
su SVHN test, MNIST test e USPS test). Qui viene ricaricato allo stesso
modo, insieme a MNIST e USPS come dataset separati per le sezioni
successive.

In [2]:
model = SmallCNN32(n_classes=ckpt["n_classes"], feature_dim=ckpt["feature_dim"])
model.load_state_dict(ckpt["state_dict"])
model.eval()
print(f"model.training = {model.training} (deve essere False)")

target_data = {}
for domain in TARGET_DOMAINS:
    X_t, y_t = load_domain(domain, "test", mean, std)
    target_data[domain] = (X_t, y_t)
    print(f"dominio target '{domain}': {X_t.shape[0]} immagini (test split, "
          f"normalizzate con le statistiche di {SOURCE_DOMAIN}-train)")

model.training = False (deve essere False)


dominio target 'mnist': 10000 immagini (test split, normalizzate con le statistiche di svhn-train)


dominio target 'usps': 2007 immagini (test split, normalizzate con le statistiche di svhn-train)


## 2. Fit di Laplace sull'ultimo layer

`tau_prior = weight_decay * N_source`: `weight_decay` è quello che
`code_v2/src/digits_train.py` ha effettivamente passato ad `AdamW` (letto dal
checkpoint, non assunto); `N_source = ckpt["n_source_train"]` è la dimensione
dello split di training interno che l'ottimizzatore ha effettivamente visto
(esclude lo split di validazione per l'early stopping).

**Il fit gira esattamente su quello stesso split, non sull'intero SVHN train.**
`theta_MAP` è il modo di

$$	ext{CE sommata sui punti di training} \;+\; 	frac{	au}{2}\lVert	hetaVert^2,
\qquad 	au = 	ext{weight\_decay}	imes N_	ext{source},$$

quindi l'Hessiana va assemblata **sugli stessi punti** che compaiono in quella
somma. Una versione precedente di questo notebook fittava sull'intero
`svhn_train.npz` (73.257 immagini) lasciando però `tau_prior` calcolato su
65.932: termine di verosimiglianza e termine di prior finivano su due $N$
diversi, e `theta_MAP` non era il modo dell'obiettivo effettivamente
approssimato. Gli indici dello split sono ora salvati nel checkpoint
(`ckpt["train_indices"]`, aggiunti da `digits_train.py`) proprio per rendere il
fit riproducibile senza dover ri-derivare la `random_split`.

Nessun sottocampionamento oltre a questo: `weight_space_hessian`
(`laplace_core.py`) costruisce l'Hessiana blocco per blocco via BLAS (`K²`
blocchi `D×D`), senza mai materializzare un tensore `(N,Dp,Dp)` -- a differenza
dell'implementazione usata in una versione precedente di questo esperimento, che
per lo stesso motivo doveva sottocampionare SVHN train a poche migliaia di
immagini.

In [3]:
tau_prior = WEIGHT_DECAY * N_SOURCE_TRAIN
print(f"tau_prior = weight_decay * N_source = {WEIGHT_DECAY} * {N_SOURCE_TRAIN} = {tau_prior:.3f}")

X_train_full, y_train_full = load_domain(SOURCE_DOMAIN, "train", mean, std)
train_idx = ckpt["train_indices"]          # split interno visto dall'ottimizzatore
X_svhn_train, y_svhn_train = X_train_full[train_idx], y_train_full[train_idx]
print(f"{SOURCE_DOMAIN} train totale:  {X_train_full.shape[0]} immagini")
print(f"pool per il fit (split di training del checkpoint): {X_svhn_train.shape[0]} immagini")
assert X_svhn_train.shape[0] == N_SOURCE_TRAIN, "il pool del fit deve coincidere con N_source di tau_prior"
print("  -> coincide con N_source usato per tau_prior: verosimiglianza e prior sulla stessa scala")

train_loader = DataLoader(TensorDataset(X_svhn_train, y_svhn_train), batch_size=256, shuffle=False)
Phi, y_np, _ = extract(model, train_loader, device="cpu")
Phi_aug = augment(Phi)
W_aug = head_weights(model)
print(f"Phi: {Phi.shape}  Phi_aug: {Phi_aug.shape}  W_aug: {W_aug.shape}")

laplace = LastLayerLaplace.fit(W_aug, Phi_aug, tau_prior=tau_prior)
print(f"\nLaplace fit: K={laplace.K}  Dp={laplace.Dp}  cov shape={laplace.cov.shape}")

map_preds = (Phi_aug @ W_aug.T).argmax(axis=1)
map_acc = (map_preds == y_np).mean()
print(f"sanity check -- MAP accuracy sulle stesse {len(y_np)} feature di training: {100 * map_acc:.2f}%")

MODELS_DIR = CHECKPOINT_PATH.parent
np.savez(MODELS_DIR / "svhn_laplace.npz", theta_map=laplace.theta_map, cov=laplace.cov,
        K=laplace.K, Dp=laplace.Dp, tau_prior=tau_prior, source_mean=mean, source_std=std)
print(f"\nsalvato {MODELS_DIR / 'svhn_laplace.npz'}")

tau_prior = weight_decay * N_source = 0.001 * 65932 = 65.932


svhn train totale:  73257 immagini
pool per il fit (split di training del checkpoint): 65932 immagini
  -> coincide con N_source usato per tau_prior: verosimiglianza e prior sulla stessa scala


Phi: (65932, 128)  Phi_aug: (65932, 129)  W_aug: (10, 129)



Laplace fit: K=10  Dp=129  cov shape=(1290, 1290)
sanity check -- MAP accuracy sulle stesse 65932 feature di training: 94.03%

salvato C:\Users\aless\PycharmProjects\BayesianExoAdaptation\code_v2\models\source_svhn\svhn_laplace.npz


## Confronto pulito source vs. target

A differenza di una versione precedente di questo esperimento (dove il
checkpoint source arrivava già pre-addestrato, senza uno split disponibile, e
ogni valutazione "source" era in realtà sui punti di training), qui il fit di
Laplace usa solo lo split interno di training di `svhn_train.npz`, mentre la
baseline source dei prossimi notebook è `svhn_test.npz`: due file disgiunti,
caricati indipendentemente. La baseline source è quindi pulita, su dati mai
visti né dal training né dal fit -- confermato di seguito controllando
esplicitamente quale split è stato usato dove.

Le 7.325 immagini di validazione (`svhn_train.npz` meno lo split di training)
non entrano né nel fit né nella baseline: sono servite solo all'early stopping.

In [4]:
X_svhn_test, y_svhn_test = load_domain(SOURCE_DOMAIN, "test", mean, std)
n_val_holdout = X_train_full.shape[0] - X_svhn_train.shape[0]
print(f"fit di Laplace:           digits/svhn_train.npz, split di training  ({X_svhn_train.shape[0]} immagini)")
print(f"validazione early stop:   digits/svhn_train.npz, split rimanente    ({n_val_holdout} immagini, non usate qui)")
print(f"baseline source (dopo):   digits/svhn_test.npz                      ({X_svhn_test.shape[0]} immagini)")
print("\nfile disgiunti, caricati indipendentemente -- train e test non possono sovrapporsi")

fit di Laplace:           digits/svhn_train.npz, split di training  (65932 immagini)
validazione early stop:   digits/svhn_train.npz, split rimanente    (7325 immagini, non usate qui)
baseline source (dopo):   digits/svhn_test.npz                      (26032 immagini)

file disgiunti, caricati indipendentemente -- train e test non possono sovrapporsi
